# CSET343 - AI in Healthcare | Lab Assignment 6
## Neural Network + Feature Preprocessing on Chest X-Ray Images (Normal vs Pneumonia)
**Author:** Bhupendra Singh


In [ ]:
import os, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.feature import hog, local_binary_pattern
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

DATA_DIR = "chest_xray"
OUT_DIR = "lab6_outputs"
IMG_SIZE = 64
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

def collect_paths(data_dir):
    paths, labels = [], []
    for p in Path(data_dir).rglob("*"):
        if p.suffix.lower() in (".jpeg", ".jpg", ".png") and p.parent.name.upper() in CLASS_NAMES:
            paths.append(str(p))
            labels.append(CLASS_NAMES.index(p.parent.name.upper()))
    return np.array(paths), np.array(labels)

paths, labels = collect_paths(DATA_DIR)
print(f"Total images collected: {len(paths)}")
print("Class counts:", pd.Series(labels).map(dict(enumerate(CLASS_NAMES))).value_counts().to_dict())

def load_image(path):
    img = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)

X_imgs = np.array([load_image(p) for p in paths])
print("Processed image array shape:", X_imgs.shape)

X_flat = X_imgs.reshape(len(X_imgs), -1) / 255.0
X_train, X_test, y_train, y_test = train_test_split(X_flat, labels, test_size=0.3, random_state=SEED, stratify=labels)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

nn_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=SEED)
nn_model.fit(X_train_scaled, y_train)
y_pred = nn_model.predict(X_test_scaled)

print("\nClassification Report (Neural Network - Chest X-Ray Pneumonia):")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Chest X-Ray Pneumonia Neural Network")
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"))
plt.show()
